---

# Create Synthetic Employee Survey Data

---

This notebook generates a synthetic employee survey dataset based on the structure of an input survey template.

Features:

- Preserves the column structure of the source template
- Generates realistic synthetic responses
- Supports multiple survey waves
- Creates synthetic free-text comments
- Anonymizes organization-specific information
- Produces reproducible results through configurable random seeds

The generated data is intended for:

- Data science projects
- Dashboard development
- Data pipeline testing
- Training and demonstration purposes

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

from datetime import datetime
from pathlib import Path

import os
import re
import random

import numpy as np
import pandas as pd

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

# Number of synthetic responses to create
N_ROWS = 500

# Survey wave configuration
RUN_COUNT = 4
RUN_INTERVAL_DAYS = 14
RUN_FIELD_DAYS = 5

# Start date of the first survey wave
BASE_DATE = datetime(2026, 1, 1, 8, 0, 0)

# Default iteration identifier if none exists
ITERATION_ID_FALLBACK = "ITERATION_01"

# Percentage of empty free-text responses
FREE_TEXT_EMPTY_SHARE = 0.05

# Optional shift applied to survey ratings
IRONIC_LIKERT_SHIFT = 1

# Generic organization placeholder
ORG_PLACEHOLDER = "[ORGANIZATION]"

# Pattern for replacing organization names if required
ORG_NAME_PATTERN = re.compile(
    r"\[ORGANIZATION_NAME\]",
    re.IGNORECASE
)

# Path to survey structure template
template_file = "./survey_template/survey_template.xlsx"

# Output folder
output_path = "./synthetic_output"

# Output file names
csv_filename = "synthetic_employee_survey.csv"
xlsx_filename = "synthetic_employee_survey.xlsx"

# Create Excel output
CREATE_EXCEL = True

random.seed(SEED)
np.random.seed(SEED)
``

In [ ]:
# ============================================================
# OPTIONAL COLUMN FILTERS
# ============================================================

# Columns that should not be included in the output

DROP_COLUMNS = [
    "[ORGANIZATION_FIELD]",
    "[BUSINESS_UNIT_FIELD]",
    "[PROJECT_MEMBERSHIP_FIELD]",
    "[SURVEY_USAGE_FIELD]",
    "[CONTEXT_FIELD]"
]

# Exclude all columns beginning with one of these prefixes

DROP_COLUMN_PREFIXES = [
    "[EXCLUDED_PREFIX]"
]

In [ ]:
# ============================================================
# SYNTHETIC FREE-TEXT RESPONSES
# ============================================================

POSITIVE_COMMENTS = [
    "The collaboration within the team works well.",
    "Communication is transparent and constructive.",
    "I have the resources required to perform my work.",
    "The working environment supports productivity.",
    "The goals and expectations are clear."
]

NEUTRAL_COMMENTS = [
    "The current situation is stable.",
    "The workload is manageable.",
    "No additional comments.",
    "The work processes are functioning adequately.",
    "No major concerns at the moment."
]

NEGATIVE_COMMENTS = [
    "Communication could be improved.",
    "Some processes are unnecessarily complex.",
    "Workload is occasionally difficult to manage.",
    "Decision-making could be faster.",
    "There are opportunities for improvement."
]

In [ ]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def normalize_column_name(col: str) -> str:
    """
    Standardize column names for matching.
    """
    return str(col).strip().lower()


def should_drop_column(column_name: str) -> bool:
    """
    Check whether a column should be removed.
    """

    normalized = normalize_column_name(column_name)

    if normalized in {
        normalize_column_name(c)
        for c in DROP_COLUMNS
    }:
        return True

    for prefix in DROP_COLUMN_PREFIXES:
        if normalized.startswith(
            normalize_column_name(prefix)
        ):
            return True

    return False

In [ ]:
# ============================================================
# LOAD TEMPLATE STRUCTURE
# ============================================================

template_df = pd.read_excel(template_file)

print("Columns found:", len(template_df.columns))

columns_to_keep = [
    col
    for col in template_df.columns
    if not should_drop_column(col)
]

print("Columns retained:", len(columns_to_keep))

In [ ]:
# ============================================================
# GENERATE SYNTHETIC DATA
# ============================================================

synthetic_df = pd.DataFrame(index=range(N_ROWS))

for column in columns_to_keep:

    col_lower = str(column).lower()

    if "comment" in col_lower:
        continue

    synthetic_df[column] = np.random.randint(
        1,
        6,
        size=N_ROWS
    )

synthetic_df.head()

In [ ]:
# ============================================================
# GENERATE SURVEY WAVE INFORMATION
# ============================================================

synthetic_df["IterationID"] = ITERATION_ID_FALLBACK

synthetic_df["RunID"] = np.random.randint(
    1,
    RUN_COUNT + 1,
    size=N_ROWS
)

In [ ]:
# ============================================================
# GENERATE SYNTHETIC COMMENTS
# ============================================================

def generate_comment():

    if random.random() < FREE_TEXT_EMPTY_SHARE:
        return np.nan

    r = random.random()

    if r < 0.33:
        return random.choice(POSITIVE_COMMENTS)

    if r < 0.66:
        return random.choice(NEUTRAL_COMMENTS)

    return random.choice(NEGATIVE_COMMENTS)


for column in synthetic_df.columns:

    if "comment" in str(column).lower():

        synthetic_df[column] = [
            generate_comment()
            for _ in range(N_ROWS)
        ]

In [ ]:
# ============================================================
# SAVE RESULTS
# ============================================================

Path(output_path).mkdir(
    parents=True,
    exist_ok=True
)

csv_path = os.path.join(
    output_path,
    csv_filename
)

xlsx_path = os.path.join(
    output_path,
    xlsx_filename
)

synthetic_df.to_csv(
    csv_path,
    index=False
)

if CREATE_EXCEL:

    synthetic_df.to_excel(
        xlsx_path,
        index=False
    )

print("CSV:", csv_path)

if CREATE_EXCEL:
    print("Excel:", xlsx_path)

## How to adapt this notebook

1. Replace `survey_template.xlsx` with your own survey structure.
2. Adjust `N_ROWS` to control dataset size.
3. Modify free-text templates if required.
4. Add custom response-generation logic for specific question types.
5. Export as CSV and/or Excel.

The notebook deliberately contains no organization-specific terminology, survey questions, department names, or proprietary business information.